# Hyderabad Mobility Atlas — Phase 1 Coverage Audit (v0.3)

**What this does:** downloads an OpenStreetMap extract covering Hyderabad and measures
*how complete the OSM data is for the specific attributes every HMA metric depends on* —
medians, turn restrictions, U-turn restrictions, crossings, lanes.

**Why this is the first thing we build:** every HMA metric (Detour Index, U-turn Burden,
Severance, Crossing Deficit) depends on attributes that are frequently missing in
peripheral Indian cities. If they are missing in Patancheru, the method does not work
there as designed and we must fix the data before publishing a single number.
This notebook is the go / no-go gate for Phase 2.

**Three zones compared:**
- `core_city` — well-mapped central Hyderabad (quality benchmark)
- `patancheru` — the pilot corridor (Patancheru / Isnapur / NH-65)
- `hmda_all` — the full HMDA study area

**Runtime:** roughly 10-20 minutes on Colab. Two passes over the extract.

**Status:** v0.3, 24 Oct 2026. The pure logic (zoning, military strip, metric maths) is
unit-tested in §7. The osmium passes could not be tested before shipping — the first run
IS that test. Expect to correct EXTRACT_URL on the first attempt.

**Legal posture:** OSM is ODbL — attribution required, share-alike on derived databases.
This notebook strips military/defence attributes on ingest (HMA Doc 01 §2.2) and asserts
their absence before writing output. No personal data is touched at any point.

## 1. Install

In [ ]:
!pip -q install osmium tabulate
!apt-get -qq install -y osmium-tool > /dev/null 2>&1
import osmium, pandas
print("osmium   :", osmium.__version__ if hasattr(osmium, "__version__") else "ok")
print("pandas   :", pandas.__version__)
import subprocess
print("osmium-tool:", subprocess.run(["osmium","--version"],capture_output=True,text=True).stdout.split("\n")[0])

## 2. Configuration

If the extract URL 404s, open https://download.geofabrik.de/asia/india.html and pick the
sub-region containing Telangana. Geofabrik splits India into zones; Telangana sits in the
southern zone. Verify this before the first run — it is the most likely thing to break.

In [ ]:
EXTRACT_URL = "https://download.geofabrik.de/asia/india/southern-zone-latest.osm.pbf"
PBF_PATH    = "/content/southern-zone.osm.pbf"

# Zones: (min_lon, min_lat, max_lon, max_lat)
ZONES = {
    "core_city":  (78.380, 17.370, 78.520, 17.460),
    "patancheru": (78.230, 17.490, 78.330, 17.570),
    "hmda_all":   (78.050, 17.050, 78.850, 17.750),
}

# Road classes that carry local access. service/track excluded from headline km.
HIGHWAY_MAIN = {
    "motorway","trunk","primary","secondary","tertiary",
    "motorway_link","trunk_link","primary_link","secondary_link","tertiary_link",
    "unclassified","residential","living_street",
}
# Arterials: the classes where median and U-turn geometry actually bites
HIGHWAY_ARTERIAL = {"motorway","trunk","primary","secondary"}

OUT_DIR = "/content/hma_out"

import os
os.makedirs(OUT_DIR, exist_ok=True)
print("configured —", len(ZONES), "zones")

## 3. Download extract

Plain Python, no shell magic, so it behaves the same on Colab, Jupyter or locally.

In [ ]:
import os, subprocess, sys

if os.path.exists(PBF_PATH) and os.path.getsize(PBF_PATH) > 1_000_000:
    print("already downloaded")
else:
    print("downloading", EXTRACT_URL)
    rc = subprocess.call(["wget", "-q", "--show-progress", "-O", PBF_PATH, EXTRACT_URL])
    if rc != 0:
        raise SystemExit(
            "Download failed. Open https://download.geofabrik.de/asia/india.html and set "
            "EXTRACT_URL to the sub-region containing Telangana.")

size_mb = os.path.getsize(PBF_PATH) / 1e6
print(round(size_mb, 1), "MB")
assert size_mb > 1, "Extract looks empty — check EXTRACT_URL."

## 3b. Clip to Hyderabad first

The southern-zone extract covers five states. Both passes below only care about the
HMDA bounding box, so clipping first cuts the work by roughly ninety percent and keeps
peak memory low enough for a free Colab instance. This is also what makes a local run
on a laptop practical.

In [ ]:
import subprocess, os

x0, y0, x1, y1 = ZONES["hmda_all"]
CLIP_PATH = "/content/hyderabad.osm.pbf"

if os.path.exists(CLIP_PATH) and os.path.getsize(CLIP_PATH) > 100_000:
    print("already clipped")
else:
    cmd = ["osmium", "extract", "-b", f"{x0},{y0},{x1},{y1}",
           PBF_PATH, "-o", CLIP_PATH, "--overwrite", "-s", "complete_ways"]
    print(" ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise SystemExit("Clip failed. If osmium-tool is missing, set CLIP_PATH = PBF_PATH "
                         "and run the passes on the full extract instead.")

before = os.path.getsize(PBF_PATH) / 1e6
after  = os.path.getsize(CLIP_PATH) / 1e6
print(f"{before:.1f} MB -> {after:.1f} MB  ({100*after/before:.1f}% of original)")

# Everything downstream reads the clipped file.
WORK_PBF = CLIP_PATH

## 4. Pure logic (unit-tested in §7)

Zone assignment, the military-attribute stripper and the metric maths live here as plain
functions with no OSM dependency, so they can be tested without parsing a 1 GB file.

In [ ]:
MILITARY_KEYS = {"military", "military_service"}
MILITARY_LANDUSE = {"military"}
MILITARY_VALUES = {"danger_area", "barracks", "naval_base", "airfield", "training_area"}

def zone_of(lon, lat, zones):
    """Every zone whose bbox contains this point. Zones overlap by design —
    core_city and patancheru both sit inside hmda_all."""
    return [name for name, (x0, y0, x1, y1) in zones.items()
            if x0 <= lon <= x1 and y0 <= lat <= y1]

def strip_military(tags):
    """HMA Doc 01 2.2 — never associate a location with a defence attribute.
    Returns (clean_tags, was_stripped). The geometry survives as an access
    restriction, which is a legitimate input to a detour calculation; the
    identity of the installation does not survive at all."""
    hit = (any(k in MILITARY_KEYS for k in tags)
           or tags.get("landuse") in MILITARY_LANDUSE
           or tags.get("military") in MILITARY_VALUES)
    if not hit:
        return tags, False
    clean = {k: v for k, v in tags.items()
             if k not in MILITARY_KEYS and k not in ("landuse", "name", "operator", "description")}
    clean["access_restricted"] = "yes"
    return clean, True

def pct(a, b):
    return round(100.0 * a / b, 1) if b else 0.0

def is_divided_proxy(tags):
    """OSM rarely carries divider= directly. A median-divided arterial is almost
    always modelled as two oneway carriageways, often with dual_carriageway=yes."""
    return tags.get("dual_carriageway") == "yes" or tags.get("oneway") == "yes"

print("logic loaded")

## 5. Pass A — restriction relations and the nodes they hinge on

Turn restrictions are OSM *relations*, which carry no coordinates of their own. Counting
them extract-wide would count all of South India, not Hyderabad — useless for a
zone-by-zone audit. So pass A collects each restriction and its `via` node reference;
pass B resolves those references to coordinates and assigns a zone.

In [ ]:
import osmium, collections

class RestrictionPass(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.via_node = {}        # node_id -> restriction value
        self.via_way  = {}        # way_id  -> restriction value
        self.no_location = collections.Counter()   # restrictions with no usable via
        self.total = 0

    def relation(self, r):
        tags = dict(r.tags)
        if tags.get("type") != "restriction":
            return
        val = tags.get("restriction") or tags.get("restriction:motorcar") or "unspecified"
        self.total += 1
        for m in r.members:
            if m.role == "via" and m.type == "n":
                self.via_node[m.ref] = val
                return
        for m in r.members:          # v0.3: recover way-based via members
            if m.role == "via" and m.type == "w":
                self.via_way[m.ref] = val
                return
        self.no_location[val] += 1   # no usable via member at all

ra = RestrictionPass()
ra.apply_file(WORK_PBF)

print("restriction relations in extract :", ra.total)
print("placeable via a via-NODE         :", len(ra.via_node))
print("placeable via a via-WAY          :", len(ra.via_way))
print("still not placeable              :", sum(ra.no_location.values()))

## 6. Pass B — ways, nodes, and resolving restrictions to zones

A way is assigned to a zone if its first, middle **or** last node falls inside it, so a
long road entering the study area is not missed because it began elsewhere.

In [ ]:
class MainPass(osmium.SimpleHandler):
    def __init__(self, via_node, via_way):
        super().__init__()
        self.via_node = via_node
        self.via_way = via_way
        self.way_rows = []
        self.node_counts = collections.Counter()
        self.zone_restrictions = collections.Counter()   # (zone, value) -> n
        self.military_stripped = 0
        self.skipped_geom = 0

    def way(self, w):
        tags = dict(w.tags)

        # v0.3: a restriction whose via member is a WAY is placed at that way's
        # first node. Done before the highway filter, since a via way can be any class.
        val = self.via_way.get(w.id)
        if val is not None:
            try:
                for z in zone_of(w.nodes[0].lon, w.nodes[0].lat, ZONES):
                    self.zone_restrictions[(z, val)] += 1
            except Exception:
                pass

        hw = tags.get("highway")
        if hw is None or hw not in HIGHWAY_MAIN:
            return

        tags, stripped = strip_military(tags)
        if stripped:
            self.military_stripped += 1

        try:
            length_m = osmium.geom.haversine_distance(w.nodes)
            pts = [w.nodes[0], w.nodes[len(w.nodes) // 2], w.nodes[-1]]
            coords = [(p.lon, p.lat) for p in pts]
        except Exception:
            self.skipped_geom += 1
            return

        zones = set()
        for lon, lat in coords:
            zones.update(zone_of(lon, lat, ZONES))
        if not zones:
            return

        self.way_rows.append({
            "zones": sorted(zones),
            "highway": hw,
            "length_m": length_m,
            "divided": is_divided_proxy(tags),
            "lanes": tags.get("lanes"),
            "surface": tags.get("surface"),
            "sidewalk": tags.get("sidewalk"),
            "named": 1 if "name" in tags else 0,
        })

    def node(self, n):
        try:
            lon, lat = n.location.lon, n.location.lat
        except Exception:
            return

        # resolve a turn restriction to a zone
        val = self.via_node.get(n.id)
        if val is not None:
            for z in zone_of(lon, lat, ZONES):
                self.zone_restrictions[(z, val)] += 1

        tags = dict(n.tags)
        if not tags:
            return
        kind = None
        if tags.get("highway") == "crossing":
            kind = "crossing"
        elif tags.get("highway") == "traffic_signals":
            kind = "traffic_signals"
        elif tags.get("barrier"):
            kind = "barrier"
        if kind:
            for z in zone_of(lon, lat, ZONES):
                self.node_counts[(z, kind)] += 1

h = MainPass(ra.via_node, ra.via_way)
h.apply_file(WORK_PBF, locations=True)

print("ways kept              :", len(h.way_rows))
print("military stripped      :", h.military_stripped)
print("geometry skipped       :", h.skipped_geom)
print("restrictions placed    :", sum(h.zone_restrictions.values()))

## 7. Tests — these run before any number is reported

Per the working constitution, a release is not done until its test exists. Two kinds here:
unit tests on the pure logic, and an assertion that no defence attribute reached output.

In [ ]:
import pandas as pd

# --- unit tests on pure logic -------------------------------------------------
Z = {"a": (0, 0, 10, 10), "b": (5, 5, 20, 20)}
assert zone_of(1, 1, Z) == ["a"]
assert sorted(zone_of(7, 7, Z)) == ["a", "b"], "overlapping zones must both match"
assert zone_of(50, 50, Z) == []

t, s = strip_military({"highway": "residential", "name": "Main Rd"})
assert s is False and t["name"] == "Main Rd"

t, s = strip_military({"highway": "service", "landuse": "military", "name": "Depot Rd"})
assert s is True
assert "landuse" not in t and "name" not in t and t["access_restricted"] == "yes"

t, s = strip_military({"highway": "service", "military": "barracks", "operator": "X"})
assert s is True and "military" not in t and "operator" not in t

assert pct(1, 4) == 25.0 and pct(0, 0) == 0.0
assert is_divided_proxy({"oneway": "yes"}) is True
assert is_divided_proxy({"dual_carriageway": "yes"}) is True
assert is_divided_proxy({"highway": "residential"}) is False
print("PASS — pure logic unit tests")

# --- build the frame ----------------------------------------------------------
rows = []
for r in h.way_rows:
    for z in r["zones"]:
        rr = {k: v for k, v in r.items() if k != "zones"}
        rr["zone"] = z
        rows.append(rr)
df = pd.DataFrame(rows)
assert not df.empty, "No roads matched any zone — check ZONES and the extract area."

# --- negative-attribute assertion --------------------------------------------
banned = {"military", "military_service", "landuse", "operator"}
leaked = banned.intersection(set(df.columns))
assert not leaked, f"NEGATIVE-ATTRIBUTE TEST FAILED: {leaked}"
print("PASS — no defence attributes in output tables")
print("rows:", len(df))

## 8. Coverage metrics — the numbers that decide Phase 2

In [ ]:
summary = []
for z in ZONES:
    d = df[df.zone == z]
    if d.empty:
        continue
    total_km = d.length_m.sum() / 1000.0
    art = d[d.highway.isin(HIGHWAY_ARTERIAL)]
    art_km = art.length_m.sum() / 1000.0
    art_div_km = art[art.divided].length_m.sum() / 1000.0

    n_cross = h.node_counts.get((z, "crossing"), 0)
    n_sig   = h.node_counts.get((z, "traffic_signals"), 0)
    n_rest  = sum(v for (zz, _), v in h.zone_restrictions.items() if zz == z)
    n_uturn = sum(v for (zz, k), v in h.zone_restrictions.items()
                  if zz == z and "u_turn" in k)

    summary.append({
        "zone": z,
        "road_km": round(total_km, 1),
        "arterial_km": round(art_km, 1),
        "arterial_divided_pct": pct(art_div_km, art_km),
        "lanes_tagged_pct": pct(d.lanes.notna().sum(), len(d)),
        "named_pct": pct(d.named.sum(), len(d)),
        "sidewalk_tagged_pct": pct(d.sidewalk.notna().sum(), len(d)),
        "crossings": n_cross,
        "crossings_per_arterial_km": round(n_cross / art_km, 2) if art_km else 0.0,
        "traffic_signals": n_sig,
        "turn_restrictions": n_rest,
        "restrictions_per_arterial_km": round(n_rest / art_km, 2) if art_km else 0.0,
        "u_turn_restrictions": n_uturn,
    })

cov = pd.DataFrame(summary)
pd.set_option("display.width", 220)
display(cov.T)

In [ ]:
# Restriction mix per zone — which kinds of turn bans are actually mapped
rest_rows = [{"zone": z, "restriction": k, "count": v}
             for (z, k), v in sorted(h.zone_restrictions.items(), key=lambda x: -x[1])]
rest = pd.DataFrame(rest_rows)
display(rest.head(30))

## 9. The gate

Compare `patancheru` against `core_city`. The absolute numbers matter less than the gap —
core city is our benchmark for what good OSM coverage looks like in Hyderabad.

| Signal | Phase 2 proceeds | Data work needed first |
|---|---|---|
| `arterial_divided_pct` gap vs core city | within ~15 points | larger gap → medians unmodelled in Patancheru |
| `crossings_per_arterial_km` (patancheru) | > 0.3 | near zero → crossings unmapped |
| `restrictions_per_arterial_km` (patancheru) | > 0.5 | near zero → turn bans unmapped |
| `u_turn_restrictions` (patancheru) | > 10 | near zero → **the core metric cannot be computed from OSM alone** |
| `lanes_tagged_pct` | > 30% | below → geometry too thin to model |

**If Patancheru fails and core city passes:** the method is sound, the data is thin.
Phase 1.5 becomes field verification plus contributing the corrections upstream to OSM.
That is a worthwhile outcome in itself and arguably the most durable public good this
project produces.

**If both fail:** turn restrictions are unmapped across Hyderabad, and the Detour Index
must be derived from geometry alone — median polylines and physical dividers — rather
than from restriction relations. Harder, still possible, and much better to know now
than after building on the assumption.

**Known limitation (v0.2):** `arterial_divided_pct` uses oneway/dual_carriageway as a
proxy for a physical median. It will over-count genuine one-way streets in the old city.
Phase 2 replaces the proxy with paired-carriageway detection.

## 10. Write outputs

In [ ]:
import datetime

stamp = datetime.date.today().isoformat()
cov.to_csv(f"{OUT_DIR}/coverage_summary_{stamp}.csv", index=False)
rest.to_csv(f"{OUT_DIR}/turn_restrictions_{stamp}.csv", index=False)

with open(f"{OUT_DIR}/RUN_NOTES_{stamp}.md", "w") as f:
    f.write(f"# HMA Phase 1 Coverage Audit — {stamp}\n\n")
    f.write(f"Extract: {EXTRACT_URL}\n\n")
    f.write(f"Ways kept: {len(h.way_rows)} | Military attributes stripped: "
            f"{h.military_stripped} | Restrictions placed: {sum(h.zone_restrictions.values())}\n\n")
    f.write("Map data (c) OpenStreetMap contributors, licensed under ODbL.\n")
    f.write("HMA derived metrics published under ODbL.\n\n")
    f.write(cov.to_markdown(index=False))
    f.write("\n\n")
    f.write(rest.head(30).to_markdown(index=False))

print("written to", OUT_DIR)
for fn in sorted(os.listdir(OUT_DIR)):
    print(" ", fn)

## 11. Save results to Drive

Writes into `Anthropic / Hyderabad Mobility Atlas / phase1_audit`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil
dest = "/content/drive/MyDrive/Anthropic/Hyderabad Mobility Atlas/phase1_audit"
os.makedirs(dest, exist_ok=True)
for fn in os.listdir(OUT_DIR):
    shutil.copy(f"{OUT_DIR}/{fn}", f"{dest}/{fn}")
print("saved to Drive:", dest)
print(sorted(os.listdir(dest)))